In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

n_samples    = 50000
n_normal     = int(n_samples * 0.98)
n_laundering = n_samples - n_normal

In [5]:
normal = pd.DataFrame({
    'transaction_id':   range(n_normal),
    'sender_account':   np.random.randint(1000, 9999, n_normal),
    'receiver_account': np.random.randint(1000, 9999, n_normal),
    'amount':           np.random.exponential(500, n_normal).round(2),
    'transaction_type': np.random.choice(['transfer', 'payment', 'deposit', 'withdrawal'], n_normal),
    'hour':             np.random.randint(8, 20, n_normal),
    'day_of_week':      np.random.randint(0, 5, n_normal),
    'num_transactions_sender':   np.random.randint(1, 10, n_normal),
    'num_transactions_receiver': np.random.randint(1, 10, n_normal),
    'same_bank':        np.random.choice([0, 1], n_normal, p=[0.3, 0.7]),
    'international':    np.random.choice([0, 1], n_normal, p=[0.9, 0.1]),
    'is_laundering':    0
})

In [10]:
normal

,transaction_id,sender_account,receiver_account,amount,transaction_type,hour,day_of_week,num_transactions_sender,num_transactions_receiver,same_bank,international,is_laundering
0,0,8270,9821,97.66,deposit,12,0,5,9,1,0,0
1,1,1860,5910,1362.43,deposit,9,0,7,6,0,1,0
2,2,6390,4985,7.21,payment,8,1,7,3,1,0,0
3,3,6191,6756,370.16,withdrawal,12,4,7,8,0,0,0
4,4,6734,8443,281.86,withdrawal,10,2,7,6,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
48995,48995,1743,4929,160.44,deposit,15,3,3,4,1,0,0
48996,48996,4282,8732,28.05,transfer,17,3,2,3,0,0,0
48997,48997,5133,8325,1009.02,withdrawal,14,0,3,9,0,0,0
48998,48998,7611,5548,151.58,transfer,9,1,1,6,0,0,0


In [12]:
laundering = pd.DataFrame({
    'transaction_id':   range(n_normal, n_samples),
    'sender_account':   np.random.randint(1000, 9999, n_laundering),
    'receiver_account': np.random.randint(1000, 9999, n_laundering),
    'amount':           np.concatenate([
                            np.random.uniform(9000, 9999, n_laundering // 2),
                            np.random.uniform(50000, 500000, n_laundering // 2)
                        ]).round(2),
    'transaction_type': np.random.choice(['transfer', 'payment'], n_laundering),
    'hour':             np.random.randint(0, 6, n_laundering),
    'day_of_week':      np.random.randint(5, 7, n_laundering),
    'num_transactions_sender':   np.random.randint(20, 100, n_laundering),
    'num_transactions_receiver': np.random.randint(20, 100, n_laundering),
    'same_bank':        np.random.choice([0, 1], n_laundering, p=[0.7, 0.3]),
    'international':    np.random.choice([0, 1], n_laundering, p=[0.3, 0.7]),
    'is_laundering':    1
})

In [13]:
df = pd.concat([normal, laundering], ignore_index=True).sample(frac=1, random_state=42)
df['amount_log'] = np.log1p(df['amount'])
df = df.reset_index(drop=True)

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib
import os

In [15]:
FEATURES = [
    "amount", "amount_log", "hour", "day_of_week",
    "num_transactions_sender", "num_transactions_receiver",
    "same_bank", "international",
    "transaction_type_payment", "transaction_type_transfer",
    "transaction_type_withdrawal"
]